# Task 1.3 — What the Paper Claims to Improve
**Paper:** DynaMMo — Li et al., KDD 2009


## Main Baseline

The paper's primary baseline is **linear interpolation** (connecting observed values with straight lines across missing gaps) and **mean substitution** (filling missing values with the column mean). The paper also compares against **GROUSE** (a competing matrix-factorisation method) and **simple AR(1) per-sequence fitting** (fitting an autoregressive model independently to each sequence without sharing latent structure).

---

## Limitation of the Baseline the Paper Identifies

Linear interpolation and mean substitution treat each sequence **independently and without any model of the underlying dynamics**. Mean substitution ignores all temporal ordering — it cannot distinguish a value that should be high (because the sequence is in an upswing) from one that should be low (because it is in a downswing). Linear interpolation assumes missing values lie on a straight line between two observed neighbours, which breaks whenever the true signal is non-monotonic (e.g., oscillating sensor readings). More critically, **both baselines use only the single sequence being imputed** — they ignore the fact that `d` co-evolving sequences share common latent drivers, which means an unobserved sequence at time `t` can still be accurately imputed using information from other sequences observed at the same timestep.

---

## How DynaMMo Overcomes This Limitation

DynaMMo fits a shared Linear Dynamical System to all `d` sequences simultaneously. The latent state `z_t` captures the common driver at time `t`. Even if sequence `j` is entirely missing at time `t`, the Kalman Filter can still estimate `z_t` from the other `d-1` sequences that are observed at `t`. The imputed value is then `x̂_{t,j} = [C·z_t^s]_j` — using cross-sequence information that no per-sequence baseline can access.

---

## Condition Under Which DynaMMo Would Not Outperform the Baseline

DynaMMo would fail to outperform linear interpolation when the **sequences are not genuinely co-evolving** — for example, if the `d` sequences are drawn from independent processes with no shared latent structure. In that case, `C` learns a meaningless projection and the latent state `z_t` does not capture useful cross-sequence information. With `k=2` latent dimensions trying to explain `d=10` independent sequences, the model is misspecified: it forces independence onto data that genuinely has it, but pays the cost of the LDS's structural constraints (linear transitions, Gaussian noise) without gaining the benefit of cross-sequence information sharing. Linear interpolation, making no such assumptions, would produce lower imputation error simply because the data are not co-evolving. This failure scenario is consistent with Assumption 3 identified in Task 1.2.
